# Inference Test — All Providers

In [7]:
from pathlib import Path
from unified_local_llm_server import LocalLLMServer
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LocalLLMServer(provider_registry=registry)

## Provider Status

In [8]:
statuses = {}
for name in server.get_providers():
    statuses[name] = await server.check_provider_by_name(name)
    ok = statuses[name]["ok"]
    url = statuses[name]["server_url"]
    print(f"  {'✓' if ok else '✗'} {name:12} {url}")

  ✓ llama_cpp    http://127.0.0.1:9090
  ✓ lm_studio    http://127.0.0.1:1234
  ✓ ollama       http://127.0.0.1:11434
  ✓ unsloth      http://127.0.0.1:8899


## Model Selection

Edit preferred models here. `None` = auto-resolve from `/v1/models`.

In [9]:
PREFERRED_MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "openai/gpt-oss-20b",
    "unsloth":   "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
    "llama_cpp": "gpt-oss-20b-MXFP4",
}

PROMPT = "Return one short sentence about why provider abstraction is useful for local LLMs."
OPTIONS = {"num_predict": 128}

## Run Inference — All Providers

Unloads each provider before switching to the next.

In [10]:
results = {}
previous_provider = None
import time
for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        print(f"[{provider}] SKIP — not reachable")
        continue

    # Unload previous before switching
    if previous_provider and previous_provider != provider:
        try:
            unloaded = server.unload_all_models(previous_provider)
            print(f"[{previous_provider}] unloaded: {unloaded}")
        except Exception as exc:
            print(f"[{previous_provider}] unload error: {exc}")

    # Resolve model
    provider_server = server.provider_server(provider)
    try:
        model = PREFERRED_MODELS.get(provider) or await provider_server.resolve_call_model(None)
    except Exception as exc:
        print(f"[{provider}] SKIP — model resolve failed: {exc}")
        continue

    print(f"\n{'='*60}")
    print(f"  {provider} / {model}")
    print(f"{'='*60}")

    llm = server.load_model(provider, model,context_length=11100)
    try:
        response = await llm.call(
            messages=[{"role": "user", "content": PROMPT}],
            options=OPTIONS,
        )
        print("###################################################")
        time.sleep(3)
        results[provider] = {"model": model, "response": response, "ok": True}
        print(f"Response: {response}")
        previous_provider = provider
    except Exception as exc:
        results[provider] = {"model": model, "error": str(exc), "ok": False}
        print(f"ERROR: {exc}")

# Cleanup
if previous_provider:
    try:
        unloaded = server.unload_all_models(previous_provider)
        print(f"\n[{previous_provider}] unloaded: {unloaded}")
    except Exception as exc:
        print(f"[{previous_provider}] unload error: {exc}")


  llama_cpp / gpt-oss-20b-MXFP4
###################################################
Response: Provider abstraction lets local LLMs swap between different
[llama_cpp] unloaded: {'unloaded': ['gpt-oss-20b-MXFP4.gguf']}

  lm_studio / openai/gpt-oss-20b
###################################################
Response: Provider abstraction lets you swap in different local LLMs without changing your code, making experimentation and deployment more flexible.
[lm_studio] unloaded: {'unloaded': ['openai/gpt-oss-20b']}

  ollama / gpt-oss:20b
###################################################
Response: Provider abstraction lets you swap between different local LLM engines or APIs without rewriting your application code.
[ollama] unloaded: {'unloaded': ['gpt-oss:20b']}

  unsloth / unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL
###################################################
Response: <think>The user wants: "Return one short sentence about why provider abstraction is useful for local LLMs." So a sentence

## Summary

In [11]:
print(f"{'Provider':<12} {'Model':<45} {'Status'}")
print("-" * 70)
for provider, r in results.items():
    status = "OK" if r["ok"] else f"FAIL: {r.get('error', '')[:30]}"
    print(f"{provider:<12} {r['model']:<45} {status}")

Provider     Model                                         Status
----------------------------------------------------------------------
llama_cpp    gpt-oss-20b-MXFP4                             OK
lm_studio    openai/gpt-oss-20b                            OK
ollama       gpt-oss:20b                                   OK
unsloth      unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL           OK
